In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

from datetime import datetime

In [0]:
%sql
SHOW TABLES IN mdo_dev_dbx.default;

database,tableName,isTemporary
default,customer_transactions_silver,false
default,metric_drift_results_delta,false
default,quality_results_delta,false
default,sql_drift_results_delta,false
,_sqldf,true


In [0]:
quality_df = spark.table(
    "mdo_dev_dbx.default.quality_results_delta"
)


metric_df = spark.table(
    "mdo_dev_dbx.default.metric_drift_results_delta"
)


sql_df = spark.table(
    "mdo_dev_dbx.default.sql_drift_results_delta"
)

In [0]:
quality_df.printSchema()

metric_df.printSchema()

sql_df.printSchema()

display(quality_df)

display(metric_df)

display(sql_df)

root
 |-- dataset_name: string (nullable = true)
 |-- rule_name: string (nullable = true)
 |-- column_name: string (nullable = true)
 |-- failed_records: long (nullable = true)
 |-- total_records: long (nullable = true)
 |-- failure_percentage: double (nullable = true)
 |-- quality_score: double (nullable = true)
 |-- status: string (nullable = true)
 |-- run_date: timestamp (nullable = true)

root
 |-- run_date: timestamp (nullable = true)
 |-- comparison_period: string (nullable = true)
 |-- metric_name: string (nullable = true)
 |-- drift_type: string (nullable = true)
 |-- old_value: double (nullable = true)
 |-- new_value: double (nullable = true)
 |-- change_percentage: double (nullable = true)
 |-- drift_score: double (nullable = true)
 |-- severity: string (nullable = true)
 |-- description: string (nullable = true)

root
 |-- change_type: string (nullable = true)
 |-- changed_component: string (nullable = true)
 |-- description: string (nullable = true)
 |-- risk_score: long (

dataset_name,rule_name,column_name,failed_records,total_records,failure_percentage,quality_score,status,run_date
customer_transactions,duplicate_transaction_check,transaction_id,0,300000,0.0,72.92801495726496,PASS,2026-07-29T09:10:22.726008Z
customer_transactions,duplicate_transaction_check,transaction_id,0,300000,0.0,72.92801495726496,PASS,2026-07-29T05:31:47.411039Z
customer_transactions,duplicate_transaction_check,transaction_id,0,300000,0.0,72.92801495726496,PASS,2026-07-28T14:48:05.627798Z
customer_transactions,duplicate_transaction_check,transaction_id,0,300000,0.0,72.92801495726496,PASS,2026-07-28T10:43:35.710458Z
customer_transactions,null_validation,all_columns,131618,300000,43.87266666666667,72.92801495726496,FAIL,2026-07-28T10:43:35.710458Z
customer_transactions,null_validation,all_columns,131618,300000,43.87266666666667,72.92801495726496,FAIL,2026-07-29T05:31:47.411039Z
customer_transactions,null_validation,all_columns,131618,300000,43.87266666666667,72.92801495726496,FAIL,2026-07-28T14:48:05.627798Z
customer_transactions,null_validation,all_columns,131618,300000,43.87266666666667,72.92801495726496,FAIL,2026-07-29T09:10:22.726008Z


run_date,comparison_period,metric_name,drift_type,old_value,new_value,change_percentage,drift_score,severity,description
2026-07-29T09:10:35.099807Z,day1_vs_day2,Numeric Features,Numeric Drift,null,null,null,14.44149274602733,LOW,Statistical change across numerical features
2026-07-29T09:10:35.099807Z,day1_vs_day2,Population Stability,PSI,null,null,null,7.559258124680385,LOW,Population distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Categorical Features,Category Drift,null,null,null,1.4552103896103894,LOW,Category distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Correlation,Correlation Drift,null,null,null,2.637989634006945,LOW,Correlation relationship change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Business KPI,KPI Drift,null,null,null,16.705572207452047,LOW,"Revenue, profit and operational KPI change"
2026-07-29T09:10:35.099807Z,day1_vs_day2,Overall,Composite,null,null,null,8.48810407063699,LOW,Weighted drift score
2026-07-29T09:10:35.099807Z,day1_vs_day2,Volume,Volume Drift,100000.0,100000.0,0.0,0.0,LOW,Daily record count change


change_type,changed_component,description,risk_score,run_date,severity,similarity_score,sql_file
METRIC_LOGIC_CHANGE,Filter Condition,Business filtering criteria changed,90,2026-07-29T09:06:33.276068Z,MEDIUM,71.72,customer_revenue_v2.sql
METRIC_LOGIC_CHANGE,Aggregation Logic,Business metric aggregation changed,90,2026-07-29T09:06:33.276061Z,HIGH,71.72,customer_revenue_v2.sql
METRIC_LOGIC_CHANGE,Column Removal,Existing column removed,90,2026-07-29T09:06:33.276072Z,HIGH,71.72,customer_revenue_v2.sql


Calculate Data Quality Score

In [0]:
def calculate_quality_score(df):

    rows = df.collect()

    completeness = 100
    validity = 100
    uniqueness = 100
    schema_score = 100


    for row in rows:

        rule = row.rule_name.lower()

        score = max(
            0,
            100 - row.failure_percentage
        )


        if "null" in rule:
            completeness = score

        elif "duplicate" in rule:
            uniqueness = score

        elif "schema" in rule:
            schema_score = score

        else:
            validity = score


    return round(
        (
            completeness
            +
            validity
            +
            uniqueness
            +
            schema_score
        ) / 4,
        2
    )

In [0]:
data_quality_score = calculate_quality_score(
    quality_df
)

print(
    "Data Quality Score:",
    data_quality_score
)

Data Quality Score: 89.03


Metric Stability Score

In [0]:
metric_score_df = (

    metric_df

    .withColumn(
        "metric_score",

        F.when(
            F.upper(F.col("severity"))=="LOW",
            90
        )

        .when(
            F.upper(F.col("severity"))=="MEDIUM",
            60
        )

        .when(
            F.upper(F.col("severity"))=="HIGH",
            30
        )

        .otherwise(100)
    )
)

In [0]:
metric_stability_score = (

    metric_score_df
    .agg(
        F.avg("metric_score")
    )
    .collect()[0][0]

)


metric_stability_score = round(
    metric_stability_score or 100,
    2
)


print(
    "Metric Stability:",
    metric_stability_score
)

Metric Stability: 90.0


SQL Stability Score

In [0]:
sql_score_df = (

    sql_df

    .withColumn(

        "sql_score",

        F.when(
            F.upper(F.col("severity"))=="LOW",
            80
        )

        .when(
            F.upper(F.col("severity"))=="MEDIUM",
            60
        )

        .when(
            F.upper(F.col("severity"))=="HIGH",
            30
        )

        .otherwise(100)

    )
)

In [0]:
sql_stability_score = (

    sql_score_df

    .agg(
        F.avg("sql_score")
    )

    .collect()[0][0]

)


sql_stability_score = round(
    sql_stability_score or 100,
    2
)


print(
    "SQL Stability:",
    sql_stability_score
)

SQL Stability: 40.0


Final Reliability Score

In [0]:
reliability_score = round(

    (data_quality_score * 0.4)

    +

    (metric_stability_score * 0.4)

    +

    (sql_stability_score * 0.2)

)

In [0]:
print(
    "Reliability Score:",
    reliability_score
)

Reliability Score: 80


Health Status

In [0]:
def get_health_status(score):

    if score >= 90:
        return "HEALTHY"

    elif score >=70:
        return "GOOD"

    elif score >=50:
        return "WARNING"

    else:
        return "CRITICAL"

In [0]:
health_status = get_health_status(
    reliability_score
)

print(
    health_status
)

GOOD


In [0]:
def generate_explanation(
    metric_df,
    sql_df,
    quality_df
):

    reasons=[]


    for row in metric_df.collect():

        if row.severity.upper()=="HIGH":

            reasons.append(
                f"{row.metric_name} drift detected"
            )


    for row in sql_df.collect():

        if row.severity.upper()=="HIGH":

            reasons.append(
                f"SQL change detected: {row.changed_component}"
            )


    for row in quality_df.collect():

        if row.failure_percentage > 10:

            reasons.append(
                f"{row.rule_name} failures increased"
            )


    if not reasons:

        return "Dataset is stable and reliable"


    return (
        "Reliability reduced because:\n- "
        +
        "\n- ".join(reasons)
    )

In [0]:
explanation = generate_explanation(
    metric_df,
    sql_df,
    quality_df
)

print(explanation)

Reliability reduced because:
- SQL change detected: Aggregation Logic
- SQL change detected: Column Removal
- null_validation failures increased
- null_validation failures increased
- null_validation failures increased
- null_validation failures increased


In [0]:
final_record = [

{
"run_date": datetime.now(),

"dataset_name":
"customer_transactions",

"data_quality_score":
data_quality_score,

"metric_stability_score":
metric_stability_score,

"sql_stability_score":
sql_stability_score,

"reliability_score":
reliability_score,

"health_status":
health_status,

"explanation":
explanation
}

]

In [0]:
reliability_df = spark.createDataFrame(
    final_record
)

display(reliability_df)

data_quality_score,dataset_name,explanation,health_status,metric_stability_score,reliability_score,run_date,sql_stability_score
89.03,customer_transactions,Reliability reduced because: - SQL change detected: Aggregation Logic - SQL change detected: Column Removal - null_validation failures increased - null_validation failures increased - null_validation failures increased - null_validation failures increased,GOOD,90.0,80,2026-07-29T09:24:47.564723Z,40.0


In [0]:
(
    reliability_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "mdo_dev_dbx.default.reliability_score_delta"
    )
)

In [0]:
%sql
SELECT *
FROM mdo_dev_dbx.default.reliability_score_delta;

data_quality_score,dataset_name,explanation,health_status,metric_stability_score,reliability_score,run_date,sql_stability_score
89.03,customer_transactions,Reliability reduced because: - SQL change detected: Aggregation Logic - SQL change detected: Column Removal - null_validation failures increased - null_validation failures increased - null_validation failures increased - null_validation failures increased,GOOD,90.0,80,2026-07-29T09:24:47.564723Z,40.0
